# VoiceIsolate Pro — ONNX Export + HuggingFace Upload

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 **Secrets** icon in the left sidebar
3. Add a secret named `HF_TOKEN` with your HuggingFace write token
   - Get one at https://huggingface.co/settings/tokens
4. Run all cells top to bottom (Runtime → Run all)

**Expected time:** ~20-30 min (demucs download + ONNX trace is the slow step)

**Output:** Three `.onnx` files uploaded to `Joker5514/models` on HuggingFace

In [ ]:
# Cell 1: Read HF_TOKEN from Colab Secrets (never hardcode it)
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN loaded from secrets ✓')

In [ ]:
# Cell 2: Install all dependencies
# torch is pre-installed on Colab; we pin the others
!pip install -q \
  onnx>=1.15.0 \
  onnxruntime>=1.17.0 \
  onnxruntime-extensions>=0.10.1 \
  huggingface-hub>=0.22.0 \
  demucs>=4.0.1 \
  tqdm>=4.66.0 \
  requests>=2.31.0

print('Dependencies installed ✓')

In [ ]:
# Cell 3: Clone VoiceIsolate-Pro repo to get the export scripts
import os
if not os.path.exists('VoiceIsolate-Pro'):
    !git clone --depth 1 https://github.com/Joker5514/VoiceIsolate-Pro.git
else:
    !git -C VoiceIsolate-Pro pull

os.chdir('VoiceIsolate-Pro')
!mkdir -p models_output
print('Repo ready, working dir:', os.getcwd())

In [ ]:
# Cell 4: Export RNNoise (fast — pure PyTorch GRU, no downloads)
# Expected output: models_output/rnnoise_suppressor.onnx (~50-200 KB)
%run scripts/export_rnnoise_onnx.py

In [ ]:
# Cell 5: Export Demucs vocals stem (slow — downloads ~300 MB weights, traces large tensor)
# Uses mdx_extra fallback (htdemucs hybrid arch is not ONNX-traceable)
# Expected output: models_output/demucs_v4_quantized.onnx (<90 MB)
%run scripts/export_demucs_onnx.py

In [ ]:
# Cell 6: Export BSRNN vocals (medium — clones vendor repo, falls back to PyTorch impl)
# Expected output: models_output/bsrnn_vocals.onnx (<50 MB)
%run scripts/export_bsrnn_onnx.py

In [ ]:
# Cell 7: Verify all three files exist before uploading
from pathlib import Path

expected = [
    'models_output/rnnoise_suppressor.onnx',
    'models_output/demucs_v4_quantized.onnx',
    'models_output/bsrnn_vocals.onnx',
]

all_ok = True
for f in expected:
    p = Path(f)
    if p.exists():
        print(f'  ✓  {f}  ({p.stat().st_size / 1024 / 1024:.1f} MB)')
    else:
        print(f'  ✗  MISSING: {f}')
        all_ok = False

if not all_ok:
    raise RuntimeError('One or more export files are missing. Fix above before uploading.')
print('\nAll files present — ready to upload ✓')

In [ ]:
# Cell 8: Upload all three models to Joker5514/models on HuggingFace
# Reads HF_TOKEN from environment (set in Cell 1)
# Creates repo if needed, uploads with 3-retry backoff, verifies CORS
# Writes models_output/manifest_sha256_patch.json when done
%run scripts/upload_models_to_huggingface.py

In [ ]:
# Cell 9: Show the manifest patch — copy these sha256 values into
# public/app/models/models-manifest.json in the VoiceIsolate-Pro repo
import json
from pathlib import Path

patch_path = Path('models_output/manifest_sha256_patch.json')
if patch_path.exists():
    data = json.loads(patch_path.read_text())
    print(json.dumps(data, indent=2))
else:
    print('manifest_sha256_patch.json not found — upload may have failed')